# Ratings Splitter Demo

This notebook creates a 70/30 per-user split for only 50 sampled users and keeps all other users fully in train.

In [27]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

import pandas as pd

from src.dataloader.ratings_splitter import RatingsSplitter, split_ratings_train_val

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
# Find project root from notebook folder.
project_root_path = Path.cwd()
if not (project_root_path / "src").exists():
    project_root_path = project_root_path.parent

ratings_input_path = project_root_path / "data" / "raw" / "ratings_train.csv"
output_directory_path = project_root_path / "data" / "processed" / "notebook_demo"
output_directory_path.mkdir(parents=True, exist_ok=True)

ratings_dataframe = pd.read_csv(ratings_input_path)
print(f"Loaded ratings rows: {len(ratings_dataframe)}")
print(f"Loaded unique users: {ratings_dataframe['userId'].nunique()}")

Loaded ratings rows: 97801
Loaded unique users: 600


In [30]:
# Build splitter with assignment settings.
# Only 50 users are sampled for validation holdout.
ratings_splitter = RatingsSplitter(
    val_fraction=0.3,
    min_interactions=2,
    seed=42,
    max_validation_users=50,
)
train_dataframe, validation_dataframe = ratings_splitter.split(ratings_dataframe)

print(f"Train rows: {len(train_dataframe)}")
print(f"Validation rows: {len(validation_dataframe)}")
print(f"Total rows kept: {len(train_dataframe) + len(validation_dataframe)}")

TypeError: RatingsSplitter.__init__() got an unexpected keyword argument 'max_validation_users'

In [ ]:
# Check split safety rules.
validation_user_ids = set(validation_dataframe["userId"].astype(int).tolist())
train_user_ids = set(train_dataframe["userId"].astype(int).tolist())

# Every validation user must also be in train.
assert validation_user_ids.issubset(train_user_ids)

# Limit validation to 50 users.
assert len(validation_user_ids) <= 50

# No row can exist in both sets.
train_pairs = set(train_dataframe[["userId", "movieId"]].itertuples(index=False, name=None))
validation_pairs = set(validation_dataframe[["userId", "movieId"]].itertuples(index=False, name=None))
assert train_pairs.isdisjoint(validation_pairs)

print(f"Users in train: {len(train_user_ids)}")
print(f"Users in validation: {len(validation_user_ids)}")
print("Validation users are subset of train users: True")

In [ ]:
# Show helper function gives same deterministic split.
function_train_dataframe, function_validation_dataframe = split_ratings_train_val(
    ratings_dataframe=ratings_dataframe,
    val_fraction=0.3,
    min_interactions=2,
    seed=42,
    max_validation_users=50,
)

print(train_dataframe.equals(function_train_dataframe))
print(validation_dataframe.equals(function_validation_dataframe))

In [ ]:
# Save split files for model training and validation.
train_output_path = output_directory_path / "ratings_train_split.csv"
validation_output_path = output_directory_path / "ratings_validation_split.csv"

train_dataframe.to_csv(train_output_path, index=False)
validation_dataframe.to_csv(validation_output_path, index=False)

print(f"Saved train split: {train_output_path}")
print(f"Saved validation split: {validation_output_path}")
train_dataframe.head(3), validation_dataframe.head(3)